# Run one experiment on Colab

This notebook holds no model or data code. It checks out the repository, downloads the public data with the repository's own command, runs one experiment config, and saves the run's manifest, scores and forecasts to Google Drive.

Before running: choose a GPU under Runtime > Change runtime type, and add a Kaggle API token to Colab's Secrets panel. Put the secret's name in `KAGGLE_SECRET` below.

In [ ]:
CONFIG = "configs/experiments/m5_validation_state_model.yaml"  # the experiment to run
REF = "main"  # branch or commit of the repository to run
DATASETS = "m5"  # datasets to download, separated by spaces
OUT = "/content/drive/MyDrive/retail-world-models/results"  # where results are saved
KAGGLE_SECRET = "KAGGLE"  # name of the Colab secret that holds the Kaggle API token

In [ ]:
import os
from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["KAGGLE_API_TOKEN"] = userdata.get(KAGGLE_SECRET)

In [ ]:
%cd /content
!rm -rf retail-world-models
!git clone -q https://github.com/kar488/retail-world-models
%cd /content/retail-world-models
!git checkout -q {REF} && git log -1 --format="running commit %H"
!pip install -q -e ".[trees,deep,fetch]"

In [ ]:
for dataset in DATASETS.split():
    !python -m rwm.data.fetch {dataset}

In [ ]:
!nvidia-smi -L
!python -m rwm.experiments.run --config {CONFIG} --strict --out "{OUT}"